In [ ]:
!pip install faster-whisper
!pip install google
!pip install -U google-genai
!pip install ipywidgets jupyterlab_widgets
!pip install python-dotenv
!pip install pydub

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

try: 
    GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]
except KeyError:
    raise SystemExit("cannot find the env variable GEMINI_API_KEY")

In [ ]:
EXTRACT_AUDIO = True

DO_TRANSCRIPT = True
TRANSCRIPT_JSON_PATH = "/transcription.json"
SAVE_TRANSCRIPT = True

LOAD_TRANSCRIPT_FROM_FILE = False

DO_LLM_INFERENCE = True

LOAD_CUT_CONFIGS_FROM_FILE = False

# gemini-3.5-flash
# gemini-2.5-flash
MODEL_NAME="gemini-3.1-flash-lite"

CLIPS_JSON_PATH = "/llm_output.json"
SAVE_LLM_OUTPUT = True

DO_TRIM = True
DO_FRAME_PREVIEW = True
DO_LAYOUT_VIDEO_TEST = True
BUILD_COMPOSITE = True
DO_EXPORT = True

In [ ]:
#BASE_PATH = "/path to project folder"
BASE_PATH = os.getcwd()

#SPEAKER_FOLDER = "/path to session recordings (speaker vide0 and presentation) (consider that you are at BASE_PATH)"
SPEAKER_FOLDER = "/clarks_session"

# ASSET LOCATIONS
ASSETS_ROOT = BASE_PATH+"/assets"

In [ ]:
import subprocess

if EXTRACT_AUDIO:
    subprocess.run([
        "ffmpeg",
        "-i", BASE_PATH + SPEAKER_FOLDER + "/video1_synced.mp4",
        "-vn",
        "-c:a", "mp3",
        "-b:a", "320k",
        BASE_PATH + SPEAKER_FOLDER + "/synced_output.mp3"
    ])

In [ ]:
from faster_whisper import WhisperModel
import json
import subprocess
import time

AUDIO_PATH = BASE_PATH + SPEAKER_FOLDER + "/synced_output.mp3"
WINDOW_SECONDS = 10 * 60  # first/last 10 minutes

def get_duration(path):
    out = subprocess.run(
        [
            "ffprobe", "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True, text=True, check=True,
    )
    return float(out.stdout.strip())

if DO_TRANSCRIPT:
    start_time = time.perf_counter()

    duration = get_duration(AUDIO_PATH)

    # Build the two clip windows (start, end) in seconds.
    # If the file is shorter than 20 min, the windows will overlap/merge —
    # clamp so we don't send bad ranges.
    first_end = min(WINDOW_SECONDS, duration)
    last_start = max(duration - WINDOW_SECONDS, first_end)

    clip_timestamps = [(0.0, first_end)]
    if last_start < duration:
        clip_timestamps.append((last_start, duration))

    # faster-whisper wants a flat list: start,end,start,end,...
    flat_clips = []
    for s, e in clip_timestamps:
        flat_clips.extend([s, e])

    model = WhisperModel("small.en", device="cpu", compute_type="int8")

    segments, info = model.transcribe(
        AUDIO_PATH,
        language="en",
        temperature=0.0,
        vad_filter=True,
        clip_timestamps=flat_clips,
    )

    structured_segments = []
    for s in segments:
        structured_segments.append({
            "start": float(s.start),
            "end": float(s.end),
            "text": s.text.strip()
        })

    result = {
        "segments": structured_segments,
        "clip_windows": clip_timestamps,
        "duration": duration,
    }

    end_time = time.perf_counter()
    elapsed = end_time - start_time

    print(json.dumps(result, indent=2))

    print("\n--- PERFORMANCE ---")
    print(f"Execution time: {elapsed:.2f} seconds")
elif LOAD_TRANSCRIPT_FROM_FILE:
    with open(BASE_PATH + SPEAKER_FOLDER + TRANSCRIPT_JSON_PATH, "r", encoding="utf-8") as f:
        result = json.load(f)

In [ ]:
import json

if SAVE_TRANSCRIPT:
    output_path = BASE_PATH + SPEAKER_FOLDER + "/transcription.json"
    
    with open(output_path, "w", encoding="utf-8") as f:
    
        json.dump(result, f, indent=2, ensure_ascii=False)
    
    print(f"Saved to {output_path}")
else:
    print("Skipping saving the transcription")

In [ ]:
import json

def build_trim_prompt(transcript_json):
    base_prompt = """
You are a high-precision video trim-point extraction system.
You are given a JSON transcript consisting of timestamped segments generated from ASR (automatic speech recognition), taken from ONLY the first and last portions of a longer video. There is a gap in the middle with no segments — this is expected and not an error. The transcript has no speaker labels, so you must infer speaker changes purely from context, phrasing, and content shifts. These segments may contain errors, repetitions, hallucinations, or mis-transcriptions.
Your task is to determine the two cut points that isolate the MAIN SPEAKER's own content, trimming everything before they start and everything after they finish — including both dead air AND other people talking.
THIS VIDEO MAY INVOLVE MULTIPLE PEOPLE:
- A host, moderator, or announcer may introduce the main speaker before they start ("please welcome...", "our next guest is...", "let's bring up...", "so today I'm joined by...")
- Another person may close out the video after the main speaker is done (thanking them, summarizing, transitioning to something else, audience Q&A wrap-up, applause callouts, "thanks so much for that, everyone give it up for...")
- The main speaker is the person who delivers the primary content/substance of the video — identify them by who is doing the actual talking/teaching/telling for the bulk of the transcript.
WHAT COUNTS AS TRIM-WORTHY AT THE START:
- Silence, room tone, or non-speech noise before anyone talks
- Technical setup / mic checks ("can you hear me", "is this recording")
- A host/moderator/announcer introducing, welcoming, or hyping up the main speaker
- Another person's own remarks, jokes, or framing before handing off to the main speaker
- False starts or throat-clearing before the main speaker's real content begins
WHAT COUNTS AS TRIM-WORTHY AT THE END:
- Silence or non-speech noise after the main speaker has finished
- A host/moderator/other person thanking, summarizing, or wrapping up after the main speaker is done
- Another person transitioning to a new segment, sponsor read, outro music, or credits
- Audience/crowd reaction callouts, applause acknowledgments made by someone other than the main speaker
- Any trailing chatter not delivered by the main speaker
WHAT MUST NEVER BE TRIMMED:
- The main speaker's own greeting/hook/intro to the audience (e.g. "Hey everyone, welcome back") IS real content — even if it sounds like a generic opener, if it is delivered by the main speaker themselves it stays in.
- The main speaker's own outro/sign-off (e.g. "Thanks for watching, see you next time") IS real content and stays in, even if a second person's outro follows it afterward.
- The main speaker's own words while responding to or interacting with a host (e.g. answering an introduction, saying "thanks for having me") from the point they start speaking substantively.
- Any substantive sentence, claim, or idea from the main speaker, even if awkwardly phrased or ASR-garbled — when in doubt, do NOT trim it.
- Never cut mid-sentence. The start cut must land on the natural boundary where the main speaker's own content begins; the end cut must land on the natural boundary where the main speaker's own content ends.
IDENTIFYING SPEAKER CHANGES WITHOUT LABELS:
Since there are no speaker labels, infer handoffs from cues such as:
- Direct address or introduction language ("please welcome X", "I'm here with X", "thanks, let's get into it")
- A shift from talking ABOUT someone/something to a first-person account or lesson (a sign the introduced person has taken over)
- A shift in tone, topic, or perspective that suggests a different person is now speaking (e.g. third-person framing giving way to first-person storytelling)
- Direct thanks or sign-off language directed AT someone ("thank you so much for that", "amazing talk", "that's all for today folks") strongly suggests it's a different, closing speaker rather than the main speaker continuing
- If a segment reads ambiguously and could belong to either speaker, prefer treating it as the MAIN speaker's content unless there is clear evidence otherwise (avoid trimming real content).
DECISION RULES:
1. Work only within the segments provided; do not invent timestamps that don't correspond to segment boundaries or clear sub-segment breaks implied by the text.
2. If ambiguous whether a passage is the host or the main speaker, prefer the LESS aggressive cut — keep more content rather than risk trimming something that belongs to the main speaker.
3. If there is no evidence of an intro/host/dead air at the start (main speaker appears to already be speaking in the very first segment), set start_cut to 0.0 or the timestamp of the first segment.
4. If there is no evidence of an outro/host/dead air at the end (main speaker is still delivering content in the very last segment), set end_cut to the end timestamp of the last segment.
5. Quote the exact evidence segment(s) you based each decision on so a human can verify quickly.
6. Provide a confidence level for each cut based on how clear the evidence is.
7. Briefly state who you inferred the main speaker to be and why, so a human can sanity-check speaker attribution.
OUTPUT FORMAT (STRICT JSON ONLY)
Return ONLY valid JSON in the following format:
{
  "main_speaker_identification": {
    "description": string,
    "reason": string
  },
  "start_cut": {
    "timestamp": float,
    "confidence": "high" | "medium" | "low",
    "evidence_segment_indices": [int, ...],
    "reason": string
  },
  "end_cut": {
    "timestamp": float,
    "confidence": "high" | "medium" | "low",
    "evidence_segment_indices": [int, ...],
    "reason": string
  }
}
"""
    return base_prompt + "\n" + json.dumps(transcript_json, indent=2)

In [ ]:
from google import genai
import json
import re

def extract_json(text: str):
    # this part removes ```json ... ``` or ``` ... ``` fences if present
    text = re.sub(r"```json\s*", "", text)
    text = re.sub(r"```", "", text)
    return json.loads(text)


if DO_LLM_INFERENCE:
    client = genai.Client(api_key=GEMINI_API_KEY)
    llm_clips = client.models.generate_content(
        model=MODEL_NAME,
        contents=build_trim_prompt(result)
    )
    print(llm_clips.text)

    data = extract_json(llm_clips.text)

elif LOAD_CUT_CONFIGS_FROM_FILE:
    with open(BASE_PATH + SPEAKER_FOLDER + CLIPS_JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

else:
    print("Nothing chosen regarding llm inference, proceeding forward")

In [ ]:
import json

if SAVE_LLM_OUTPUT:
    data = {
        "text": llm_clips.text
    }
    
    with open(BASE_PATH + SPEAKER_FOLDER + CLIPS_JSON_PATH, "w", encoding="utf-8") as f:
    
        json.dump(data, f, indent=2, ensure_ascii=False)
    
    print(f"Saved LLM output JSON to {CLIPS_JSON_PATH}")
else:
    print(f"Skipped saveing LLM output")

In [ ]:
import subprocess
import os
import json

# This script is run FROM the speaker's own folder (where
# video1_synced.mp4 / video2_synced.mp4 / speaker_banner.png live),
# so speaker-specific paths stay relative.

SOURCE_SPEAKER = BASE_PATH + SPEAKER_FOLDER + "/video1_synced.mp4"        # speaker feed (untrimmed)
SOURCE_PRESENTATION = BASE_PATH + SPEAKER_FOLDER + "/video2_synced.mp4"   # slides / screen share (untrimmed)


LOGO_PATH = os.path.join(ASSETS_ROOT, "logo.png")


OUTPUT_PATH = BASE_PATH + SPEAKER_FOLDER + "/composited_output.mp4"

# Canvas matched to reference layout proportions
CANVAS_W, CANVAS_H = 1920, 1080

# Presentation panel — left side, tall box
PRES_W, PRES_H = 1301, 745
PRES_X, PRES_Y = 19, 104

# Camera box, right side, next to presentation (not overlapping)
CAM_W, CAM_H = 574, 481
CAM_X = 1329
CAM_Y = 104

# Logo below camera box
LOGO_W = 500
LOGO_X = CAM_X + (CAM_W - LOGO_W) // 2 + 10
LOGO_Y = CAM_Y + CAM_H + 60

PRES_ZOOM = 1.0

In [ ]:
import json

if DO_TRIM:
    trim_data = extract_json(data["text"])
    
    MIN_CONFIDENCE_TO_AUTO_APPLY = "medium"  # "high" | "medium" | "low"
    CONFIDENCE_RANK = {"low": 0, "medium": 1, "high": 2}
    
    start_cut = trim_data["start_cut"]
    end_cut = trim_data["end_cut"]
    
    TRIM_START = float(start_cut["timestamp"])
    TRIM_END = float(end_cut["timestamp"])
    
    # TRIM_START = 0
    # TRIM_END = 600

    
    TRIM_DURATION = TRIM_END - TRIM_START
    
    print(f"Main speaker identified as: {trim_data['main_speaker_identification']['description']}")
    print(f"  reason: {trim_data['main_speaker_identification']['reason']}")
    print(f"\nStart cut: {TRIM_START:.2f}s (confidence: {start_cut['confidence']})")
    print(f"  reason: {start_cut['reason']}")
    print(f"\nEnd cut: {TRIM_END:.2f}s (confidence: {end_cut['confidence']})")
    print(f"  reason: {end_cut['reason']}")
    print(f"\nResulting duration: {TRIM_DURATION:.2f}s")
    
    if TRIM_END <= TRIM_START:
        raise ValueError(f"end_cut ({TRIM_END}) is not after start_cut ({TRIM_START}) — refusing to proceed.")
    
    TRIM_LOW_CONFIDENCE = (
        CONFIDENCE_RANK[start_cut["confidence"]] < CONFIDENCE_RANK[MIN_CONFIDENCE_TO_AUTO_APPLY]
        or CONFIDENCE_RANK[end_cut["confidence"]] < CONFIDENCE_RANK[MIN_CONFIDENCE_TO_AUTO_APPLY]
    )
    
    if TRIM_LOW_CONFIDENCE:
        print(f"\n[HOLD] One or both cuts are below '{MIN_CONFIDENCE_TO_AUTO_APPLY}' confidence.")
        print("Review reasons/evidence above. DO_EXPORT will still run below unless you gate it on TRIM_LOW_CONFIDENCE yourself.")
    
    # Optional: still save a manifest of the *decision*, just not of cut files
    with open(BASE_PATH + SPEAKER_FOLDER + "/trim_manifest.json", "w", encoding="utf-8") as f:
        json.dump({
            "start": TRIM_START,
            "end": TRIM_END,
            "duration": TRIM_DURATION,
            "start_cut": start_cut,
            "end_cut": end_cut,
            "main_speaker_identification": trim_data["main_speaker_identification"],
        }, f, indent=2)

else:

    from pydub import AudioSegment

    audio = AudioSegment.from_mp3(BASE_PATH + SPEAKER_FOLDER + "/synced_output.mp3")
    
    TRIM_START = 0
    TRIM_END = len(audio) / 1000.0
    
    TRIM_DURATION = TRIM_END - TRIM_START    

In [ ]:
def build_filter_complex(bg_duration, fps=None):
    zoom_w = int(PRES_W * PRES_ZOOM)
    zoom_h = int(PRES_H * PRES_ZOOM)

    fps_filter = f",fps={fps}" if fps else ""

    return (
        f"color=c=0xFFFAEC:s={CANVAS_W}x{CANVAS_H}:d={bg_duration}[bg];"
        f"[1:v]setpts=PTS-STARTPTS,"
        f"scale={zoom_w}:{zoom_h}:force_original_aspect_ratio=increase,"
        f"crop={PRES_W}:{PRES_H}[pres];"
        f"[0:v]setpts=PTS-STARTPTS,"
        f"scale={CAM_W}:{CAM_H}:force_original_aspect_ratio=increase,"
        f"crop={CAM_W}:{CAM_H}[cam];"
        f"[2:v]scale={LOGO_W}:-1[logo];"
        f"[bg][pres]overlay={PRES_X}:{PRES_Y}[step1];"
        f"[step1][cam]overlay={CAM_X}:{CAM_Y}[step2];"
        f"[step2][logo]overlay={LOGO_X}:{LOGO_Y}{fps_filter}[outv]"
    )


def build_layout(reencode_preset="veryfast", fps=None, sample_rate=None, channels=None):
    filter_complex = build_filter_complex(TRIM_DURATION, fps=fps)

    cmd = [
        "ffmpeg", "-y",
        "-ss", str(TRIM_START), "-t", str(TRIM_DURATION), "-i", SOURCE_SPEAKER,        # input 0
        "-ss", str(TRIM_START), "-t", str(TRIM_DURATION), "-i", SOURCE_PRESENTATION,   # input 1
        "-loop", "1", "-i", LOGO_PATH,                                                 # input 2
        "-filter_complex", filter_complex,
        "-map", "[outv]",
        "-map", "0:a",
        "-c:v", "libx264",
        "-preset", reencode_preset,
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
    ]
    if sample_rate:
        cmd += ["-ar", str(sample_rate)]
    if channels:
        cmd += ["-ac", str(channels)]
    cmd += [
        "-shortest",
        OUTPUT_PATH,
    ]
    return cmd

In [ ]:
from pathlib import Path

def build_preview_cmd(timestamp_in_trimmed_clip, out_path):
    filter_complex = build_filter_complex(1)
    real_ts_speaker = TRIM_START + timestamp_in_trimmed_clip
    return [
        "ffmpeg", "-y",
        "-ss", str(real_ts_speaker), "-i", SOURCE_SPEAKER,
        "-ss", str(real_ts_speaker), "-i", SOURCE_PRESENTATION,
        "-loop", "1", "-i", LOGO_PATH,
        "-filter_complex", filter_complex,
        "-map", "[outv]",
        "-frames:v", "1",
        "-update", "1",
        out_path,
    ]


if DO_FRAME_PREVIEW:
    PREVIEW_TIMESTAMP = 1.5
    PREVIEW_PATH = BASE_PATH + SPEAKER_FOLDER + "/preview.jpg"

    cmd = build_preview_cmd(PREVIEW_TIMESTAMP, PREVIEW_PATH)
    subprocess.run(cmd, capture_output=True, text=True)

In [ ]:
def build_layout_test(reencode_preset="veryfast", test_duration=10):
    capped = min(test_duration, TRIM_DURATION)
    filter_complex = build_filter_complex(capped)
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(TRIM_START), "-t", str(capped), "-i", SOURCE_SPEAKER,
        "-ss", str(TRIM_START), "-t", str(capped), "-i", SOURCE_PRESENTATION,
        "-loop", "1", "-i", LOGO_PATH,
        "-filter_complex", filter_complex,
        "-map", "[outv]",
        "-map", "0:a",
        "-c:v", "libx264",
        "-preset", reencode_preset,
        "-c:a", "aac",
        "-shortest",
        BASE_PATH + SPEAKER_FOLDER + "/test_output.mp4",
    ]
    return cmd

if DO_LAYOUT_VIDEO_TEST:
    cmd = build_layout_test()
    layout = subprocess.run(cmd, capture_output=True, text=True)
    print("Return code:", layout.returncode)
    if layout.returncode != 0:
        print(layout.stderr[-2000:])
    else:
        print("test_output.mp4 ready for review")

In [ ]:
# ── VIDEO SEQUENCE (in final concat order)
# 1. Intro banner      (5s, static image, shared asset)
# 2. Main composited talk (presentation + speaker cam + logo)

INTRO_BANNER_PATH = os.path.join(BASE_PATH + SPEAKER_FOLDER + "/speaker_banner.png")     # shared, 5s

INTRO_BANNER_DURATION = 5
BANNER_DURATION = 5

FINAL_OUTPUT_PATH = BASE_PATH + SPEAKER_FOLDER + "/final_output.mp4"

INTRO_BANNER_SEGMENT = BASE_PATH + SPEAKER_FOLDER + "/intro_banner_segment.mp4"
COMPOSITE_SEGMENT = BASE_PATH + SPEAKER_FOLDER + "/composite_segment.mp4"

In [ ]:
import subprocess

def get_video_fps(path):
    out = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=avg_frame_rate",
         "-of", "default=noprint_wrappers=1:nokey=1", path],
        capture_output=True, text=True, check=True,
    )
    return out.stdout.strip()
    

FINAL_FPS = "30"
INTRO_FPS = "30"
BANNER_FPS = "30"

# Shared audio format for every segment that gets concatenated.
FINAL_SAMPLE_RATE = 44100
FINAL_CHANNELS = 2

def build_banner_cmd(image_path, out_path, duration=None, fps=None, sample_rate=None, channels=None):
    duration = duration or BANNER_DURATION
    fps = fps or FINAL_FPS
    sample_rate = sample_rate or FINAL_SAMPLE_RATE
    channels = channels or FINAL_CHANNELS
    channel_layout = "stereo" if channels == 2 else "mono"
    return [
        "ffmpeg", "-y",
        "-loop", "1", "-t", str(duration), "-i", image_path,
        "-f", "lavfi", "-t", str(duration), "-i", f"anullsrc=channel_layout={channel_layout}:sample_rate={sample_rate}",
        "-vf", (
            f"scale={CANVAS_W}:{CANVAS_H}:force_original_aspect_ratio=decrease,"
            f"pad={CANVAS_W}:{CANVAS_H}:(ow-iw)/2:(oh-ih)/2:color=black,"
            f"setsar=1,fps={fps}"
        ),
        "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-ar", str(sample_rate), "-ac", str(channels),
        "-shortest",
        out_path,
    ]

def build_normalize_cmd(in_path, out_path, fps=None, sample_rate=None, channels=None):
    fps = fps or FINAL_FPS
    sample_rate = sample_rate or FINAL_SAMPLE_RATE
    channels = channels or FINAL_CHANNELS
    return [
        "ffmpeg", "-y", "-i", in_path,
        "-vf", (
            f"scale={CANVAS_W}:{CANVAS_H}:force_original_aspect_ratio=decrease,"
            f"pad={CANVAS_W}:{CANVAS_H}:(ow-iw)/2:(oh-ih)/2:color=black,"
            f"setsar=1,fps={fps}"
        ),
        "-c:v", "libx264", "-preset", "veryfast", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-ar", str(sample_rate), "-ac", str(channels),
        out_path,
    ]

In [ ]:
import sys

def run_ffmpeg_with_progress(cmd, label=""):
    """Run an ffmpeg command, streaming its progress line to stdout
    instead of buffering everything until the process exits."""
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    stderr_lines = []
    for line in process.stdout:
        stderr_lines.append(line)
        line = line.strip()
        # ffmpeg emits progress on lines starting with "frame="
        if line.startswith("frame="):
            print(f"\r{label} {line}", end="", flush=True)
    process.wait()
    if any(l.strip().startswith("frame=") for l in stderr_lines):
        print()  # newline after the last carriage-returned progress line
    return process.returncode, "".join(stderr_lines)

if DO_EXPORT:
    print("Building composite layout...")
    if BUILD_COMPOSITE:
        layout_cmd = build_layout(sample_rate=FINAL_SAMPLE_RATE, channels=FINAL_CHANNELS)
        returncode, stderr = run_ffmpeg_with_progress(layout_cmd, label="[composite]")
        if returncode != 0:
            raise RuntimeError(f"ffmpeg failed on composite layout:\n{stderr[-2000:]}")

    steps = [
        (build_banner_cmd(INTRO_BANNER_PATH, INTRO_BANNER_SEGMENT, duration=INTRO_BANNER_DURATION, fps=BANNER_FPS), INTRO_BANNER_SEGMENT),
    ]
    for cmd, label in steps:
        print(f"Building {label} ...")
        returncode, stderr = run_ffmpeg_with_progress(cmd, label=f"[{label}]")
        if returncode != 0:
            raise RuntimeError(f"ffmpeg failed on {label}:\n{stderr[-2000:]}")

    all_segments = steps + [(None, OUTPUT_PATH)]

    # Save a manifest of the concat order, just for reference
    concat_list_path = BASE_PATH + SPEAKER_FOLDER + "/final_concat_list.txt"
    with open(concat_list_path, "w") as f:
        for _, seg in all_segments:
            f.write(f"file \'{seg}\'\n")

    # 2 inputs now: intro banner, main composite
    concat_cmd = [
        "ffmpeg", "-y",
        "-i", INTRO_BANNER_SEGMENT,
        "-i", OUTPUT_PATH,
        "-filter_complex",
        (
            "[0:v]setpts=PTS-STARTPTS[v0];[0:a]asetpts=PTS-STARTPTS[a0];"
            "[1:v]setpts=PTS-STARTPTS[v1];[1:a]asetpts=PTS-STARTPTS[a1];"
            "[v0][a0][v1][a1]"
            "concat=n=2:v=1:a=1[outv][outa]"
        ),
        "-map", "[outv]",
        "-map", "[outa]",
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-pix_fmt", "yuv420p",
        "-c:a", "aac",
        "-ar", str(FINAL_SAMPLE_RATE),
        "-ac", str(FINAL_CHANNELS),
        FINAL_OUTPUT_PATH,
    ]
    print("Concatenating segments...")
    returncode, stderr = run_ffmpeg_with_progress(concat_cmd, label="[concat]")
    if returncode != 0:
        raise RuntimeError(f"ffmpeg concat failed:\n{stderr[-2000:]}")
    else:
        print(f"Done. Final video: {FINAL_OUTPUT_PATH}")